In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

fatal: destination path 'RecSys_Course_AT_PoliMi' already exists and is not an empty directory.
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

Tensorflow is not available


In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [5]:
#df_train = df_train.iloc[:-1]

In [6]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [7]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))




In [8]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [9]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri 
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        recommender_instance = FeatureCombinedImplicitALSRecommender(URM_combined)
        recommender_instance.fit(
            iterations = optuna_trial.suggest_int("iterations", 100, 200),
            factors = optuna_trial.suggest_int("factors", 75, 150),
            alpha = optuna_trial.suggest_float('alpha', 5, 10),
            regularization = optuna_trial.suggest_float("regularization", 1e-5, 1e-2),
            )
        
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)

#def objective_function_funksvd(optuna_trial):
#    start_time = time.time()
#    try:
#        scores = []
#        for i in range(5):
 #           URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
 #           recommender_instance = RP3betaRecommender(URM_combined)
 #           recommender_instance.fit(alpha = optuna_trial.suggest_float("alpha", 1e-4, 9e-1, log=True),
 #                            beta = optuna_trial.suggest_float("beta", 1e-4, 9e-1, log=True),
  #                           topK =  optuna_trial.suggest_int("topK", 1, 100),
 #                            )
 #           evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[10])
 #           result, _ = evaluator_test.evaluateRecommender(recommender_instance)
 #           scores.append(result["MAP"].values[0])
 #   except Exception as e:
 #       print(f"Error during trial execution: {e}")
 #      scores = [0] * 5  # Default scores in case of failure
  #  finally:
  #      optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time) / 60)
    
 #   return sum(scores) / len(scores)

In [ ]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 50)

[I 2025-12-29 21:33:09,606] A new study created in memory with name: no-name-6bb2b870-8e5d-4019-8f4c-b6d9a00fed7f


EvaluatorHoldout: Ignoring 25 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27070 (100.0%) in 4.78 sec. Users per second: 5660
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27064 (100.0%) in 4.84 sec. Users per second: 5597
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 4.80 sec. Users per second: 5640
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 4.80 sec. Users per second: 5641
EvaluatorHoldout: Ignoring 42 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27053 (100.0%) in 4.80 sec. Users per second: 5637


[I 2025-12-29 21:48:09,554] Trial 0 finished with value: 0.24197302185244957 and parameters: {'iterations': 146, 'factors': 133, 'alpha': 7.468323499685922, 'regularization': 0.00865196485829415}. Best is trial 0 with value: 0.24197302185244957.


[0.2415884254938026, 0.24324029126720956, 0.24232047337455517, 0.24106406631198019, 0.24165185281470045]
EvaluatorHoldout: Ignoring 25 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27070 (100.0%) in 4.74 sec. Users per second: 5709
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27064 (100.0%) in 4.72 sec. Users per second: 5732
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 4.71 sec. Users per second: 5743
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 4.78 sec. Users per second: 5660
EvaluatorHoldout: Ignoring 42 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27053 (100.0%) in 4.73 sec. Users per second: 5725


[I 2025-12-29 21:57:33,315] Trial 1 finished with value: 0.24390644234885253 and parameters: {'iterations': 120, 'factors': 101, 'alpha': 6.378642974082695, 'regularization': 0.004817839235351739}. Best is trial 1 with value: 0.24390644234885253.


...

[0.24495250741397026, 0.2463082032396468, 0.24566736630116648, 0.24433647077460585, 0.24510924721505198]
EvaluatorHoldout: Ignoring 25 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27070 (100.0%) in 4.77 sec. Users per second: 5673
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27064 (100.0%) in 4.76 sec. Users per second: 5682
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 4.80 sec. Users per second: 5631
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 4.78 sec. Users per second: 5666
EvaluatorHoldout: Ignoring 42 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27053 (100.0%) in 4.72 sec. Users per second: 5730


[I 2025-12-30 01:27:32,840] Trial 18 finished with value: 0.24594273518208673 and parameters: {'iterations': 180, 'factors': 87, 'alpha': 7.580023587495706, 'regularization': 0.007358855017990155}. Best is trial 15 with value: 0.24599443237265756.


[0.24550215680392956, 0.24690101205415074, 0.24645429439240168, 0.2447889340301498, 0.2460672786298019]
EvaluatorHoldout: Ignoring 25 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27070 (100.0%) in 4.78 sec. Users per second: 5659
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27064 (100.0%) in 4.83 sec. Users per second: 5600
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 4.89 sec. Users per second: 5534
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27056 (100.0%) in 4.85 sec. Users per second: 5582
EvaluatorHoldout: Ignoring 42 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27053 (100.0%) in 4.83 sec. Users per second: 5601


[I 2025-12-30 01:43:17,758] Trial 19 finished with value: 0.24333499872310488 and parameters: {'iterations': 184, 'factors': 120, 'alpha': 7.291466082707139, 'regularization': 0.007909404754046967}. Best is trial 15 with value: 0.24599443237265756.


[0.24319722439606067, 0.2447338591854051, 0.2431293101385409, 0.2417552439745614, 0.24385935592095623]
EvaluatorHoldout: Ignoring 25 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27070 (100.0%) in 4.79 sec. Users per second: 5652
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27064 (100.0%) in 4.77 sec. Users per second: 5669
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27050 (100.0%) in 4.92 sec. Users per second: 5494
EvaluatorHoldout: Ignoring 39 ( 0.1%) Users that have less than 1 test interactions


In [ ]:
optuna_study.best_trial.params

In [ ]:
save_results.results_df

In [ ]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams